In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
# uncomment to use hosted db
#del os.environ["BIRDDOG_USE_LOCAL_NOCODB"]
if os.environ.get("BIRDDOG_USE_LOCAL_NOCODB"):
    print("using local db")
else:
    print("using AWS db")

using local db


In [3]:
from datetime import datetime

from birddog.database import Database
from birddog.database_dashboard import (
    get_doc_id_by_process_code,
    assign_docs_to_pages,
    split_doc_list_by_code,
    update_opus_summary,
)

2026-05-09 16:15:37,380 [INFO] Using local nocodb api: http://localhost:8080


In [4]:
from contextlib import contextmanager
import time

@contextmanager
def timer(label="Elapsed"):
    start = time.perf_counter()
    try:
        yield
    finally:
        print(f"{label}: {time.perf_counter() - start:.3f}s")

In [5]:
db = Database()

2026-05-09 16:15:39,747 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  localhost:api                         20.00    38.22    39.00       0.00           24


In [6]:
with timer():
    dids = get_doc_id_by_process_code(db, ["P1", "P2", "FX"])

Elapsed: 0.292s


In [7]:
len(dids)

2662

In [8]:
all_dids = list(dids.keys())

In [23]:
with timer():
    dm, pm = assign_docs_to_pages(db, all_dids)

2026-05-09 16:33:38,337 [INFO] _make_doc_map: loading doc records (2662)
2026-05-09 16:33:39,639 [INFO] _make_page_tree: loading owning page records (2638)
2026-05-09 16:33:40,687 [INFO] _make_page_tree: loading parent page records (176)
2026-05-09 16:33:40,771 [INFO] _make_page_tree: loading parent page records (153)
2026-05-09 16:33:40,849 [INFO] _make_page_tree: loading parent page records (3)
Elapsed: 2.600s


In [24]:
list(dm.items())[:5]

[(13,
  {'owning_pages': [1010],
   'process_code': 'P1',
   'processed': 1,
   'pages_processed': 123,
   'transcribed': 0,
   'assigned_page': 1010}),
 (22,
  {'owning_pages': [2066],
   'process_code': 'P1',
   'processed': 1,
   'pages_processed': 267,
   'transcribed': 0,
   'assigned_page': 2066}),
 (23,
  {'owning_pages': [1104],
   'process_code': 'P1',
   'processed': 1,
   'pages_processed': 678,
   'transcribed': 0,
   'assigned_page': 1104}),
 (25,
  {'owning_pages': [181],
   'process_code': 'P1',
   'processed': 1,
   'pages_processed': 355,
   'transcribed': 0,
   'assigned_page': 181}),
 (30,
  {'owning_pages': [1333],
   'process_code': 'P2',
   'processed': 1,
   'pages_processed': 37,
   'transcribed': 0,
   'assigned_page': 1333})]

In [25]:
list(pm.items())[:5]

[(8192,
  {'level': 'case',
   'label': 'DAKIRO-D/216/1/86',
   'url': 'https://uk.wikisource.org/wiki/Архів:ДАКрО/216/1/86',
   'parent': [8119],
   'assigned_docs': [2327],
   'assigned_parent': 8119}),
 (8196,
  {'level': 'case',
   'label': 'DAKIRO-D/195/1/15',
   'url': 'https://uk.wikisource.org/wiki/Архів:ДАКрО/195/1/15',
   'parent': [7643],
   'assigned_docs': [2792],
   'assigned_parent': 7643}),
 (8198,
  {'level': 'case',
   'label': 'DAKIRO-D/223/1/5',
   'url': 'https://uk.wikisource.org/wiki/Архів:ДАКрО/223/1/5',
   'parent': [9014],
   'assigned_docs': [3992],
   'assigned_parent': 9014}),
 (8206,
  {'level': 'case',
   'label': 'DAKIRO-D/78/1/1269',
   'url': 'https://uk.wikisource.org/wiki/Архів:ДАКрО/78/1/1269',
   'parent': [6535],
   'assigned_docs': [4225],
   'assigned_parent': 6535}),
 (8208,
  {'level': 'case',
   'label': 'DAKIRO-D/211/1/1',
   'url': 'https://uk.wikisource.org/wiki/Архів:ДАКрО/211/1/1',
   'parent': [8775],
   'assigned_docs': [3363],
   'ass

In [26]:
from graphlib import TopologicalSorter

def topo_sort_records(records: dict) -> list:
    """
    Returns record keys in topological order where each record
    appears before its assigned_parent.
    """
    graph = {}
    for k, v in records.items():
        parent = v.get('assigned_parent')
        if parent is not None and parent in records:
            # k must come before parent → parent depends on k
            graph.setdefault(parent, set()).add(k)
        graph.setdefault(k, set())  # ensure isolated nodes are included

    ts = TopologicalSorter(graph)
    return list(ts.static_order())

In [27]:
def _accumulate(record, code, delta):
    accumulator = record.get(code, {})
    for k, v in delta.items():
        accumulator[k] = accumulator.get(k, 0) + v
    record[code] = accumulator
    return record

In [28]:
def eval_summary(page_map, doc_map):
    for doc_id, doc in doc_map.items():
        code = doc["process_code"]
        sums = {
            "files": 1,
            "files_processed": doc["processed"],
            "pages_processed": doc["pages_processed"],
            "files_transcribed": doc["transcribed"],
        }
        page = page_map[doc["assigned_page"]]
        accum_sums = page.get("sums", {})
        _accumulate(accum_sums, code, sums)
        page["sums"] = accum_sums

    if True:
        for page_id in topo_sort_records(page_map):
            page_rec = page_map[page_id]
            parent_id = page_rec.get("assigned_parent")
            if parent_id:
                parent_rec = page_map[parent_id]
                parent_sums = parent_rec.get("sums", {})
                page_sums = page_rec.get("sums", {})
                for code, values in page_sums.items():
                    _accumulate(parent_sums, code, values)
                parent_rec["sums"] = parent_sums
            

In [29]:
eval_summary(pm, dm)

In [30]:
codes = [ "P1", "P2", "FX" ]

In [31]:
for pid, rec in list(pm.items())[:50]:
    #if rec.get("assigned_parent"):
    #    continue
    report = f"{pid}: {rec['label']}: "
    sums = rec.get("sums", {})
    if not sums:
        continue
    for code in codes:
        s = sums.get(code, {})
        a = [ str(s.get(k, 0)) for k in [ "files", "files_processed", "pages_processed", "files_transcribed" ] ]
        report += f"{code}: {','.join(a)}; "
    print(report)

8192: DAKIRO-D/216/1/86: P1: 1,1,175,0; P2: 0,0,0,0; FX: 0,0,0,0; 
8196: DAKIRO-D/195/1/15: P1: 1,1,145,0; P2: 0,0,0,0; FX: 0,0,0,0; 
8198: DAKIRO-D/223/1/5: P1: 1,1,80,0; P2: 0,0,0,0; FX: 0,0,0,0; 
8206: DAKIRO-D/78/1/1269: P1: 1,1,233,0; P2: 0,0,0,0; FX: 0,0,0,0; 
8208: DAKIRO-D/211/1/1: P1: 1,1,42,0; P2: 0,0,0,0; FX: 0,0,0,0; 
8225: DAKIRO-D/403/1/48: P1: 1,1,26,1; P2: 0,0,0,0; FX: 0,0,0,0; 
8226: DAKIRO-D/225/1/131: P1: 0,0,0,0; P2: 1,1,5,0; FX: 0,0,0,0; 
8227: DAKIRO-D/216/1/72: P1: 1,1,197,0; P2: 0,0,0,0; FX: 0,0,0,0; 
8230: DAKIRO-D/226/1/1: P1: 1,1,101,0; P2: 0,0,0,0; FX: 0,0,0,0; 
8234: DAKIRO-D/219/1/72: P1: 1,1,265,0; P2: 0,0,0,0; FX: 0,0,0,0; 
45: DAVIO-R/R-4262/1/5: P1: 1,1,53,0; P2: 0,0,0,0; FX: 0,0,0,0; 
8238: DAKIRO-D/216/1/67: P1: 1,1,332,0; P2: 0,0,0,0; FX: 0,0,0,0; 
8240: DAKIRO-D/211/1/7: P1: 1,1,89,0; P2: 0,0,0,0; FX: 0,0,0,0; 
8247: DAKIRO-D/225/1/291: P1: 1,1,218,0; P2: 0,0,0,0; FX: 0,0,0,0; 
8250: DAKIRO-D/216/1/29: P1: 1,1,236,0; P2: 0,0,0,0; FX: 0,0,0,0; 
8251